In [1]:
!pip install -q numpy==1.26.4 # для совместимости с fasttext
# Устанавливаем fasttext (версию wheel, она стабильнее в Colab)
!pip install -q fasttext-wheel

# Скачиваем официальную модель для определения языка (около 130 МБ)
import os
if not os.path.exists('lid.176.bin'):
    !wget -q https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
    print("Модель FastText скачана.")
else:
    print("Модель FastText уже существует.")

import hashlib
import nltk
from nltk.tokenize import sent_tokenize

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    # Для новых версий NLTK иногда нужен еще punkt_tab
    try:
        nltk.download('punkt_tab')
    except:
        pass

print("NLTK готов.")

Модель FastText уже существует.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...


NLTK готов.


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
import re
import fasttext
import requests

# 1. Загружаем модель FastText
# Отключаем предупреждения fasttext
fasttext.FastText.eprint = lambda x: None
print("Загрузка модели FastText...")
lang_model = fasttext.load_model('lid.176.bin')
print("Модель загружена.")

# 2. Загрузка списка плохих слов
BAD_WORDS_URL = "https://raw.githubusercontent.com/LDNOOBW/List-of-Dirty-Naughty-Obscene-and-Otherwise-Bad-Words/master/en"
response = requests.get(BAD_WORDS_URL)
BAD_WORDS = set(response.text.splitlines()) if response.status_code == 200 else set()
print(f"Загружено запрещенных слов: {len(BAD_WORDS)}")

# 3. Шаблоны для удаления мусора (Boilerplate)
BOILERPLATE_PATTERNS = [
    "terms of use", "privacy policy", "cookie policy",
    "uses cookies", "use of cookies", "use cookies",
    "terms and conditions",  # <-- Добавлено
    "all rights reserved",   # <-- Добавлено
    "see ap.org",            # <-- Добавлено (специфика AP)
]

# Регулярки
CITATION_REGEX = re.compile(r'\[\d+\]|\[citation needed\]', re.IGNORECASE)
LOREM_IPSUM_REGEX = re.compile(r'lorem ipsum', re.IGNORECASE)

Загрузка модели FastText...
Модель загружена.
Загружено запрещенных слов: 403


In [3]:
def is_english(text, threshold=0.80):
    """
    Проверяет, является ли текст английским.
    threshold: минимальная уверенность модели (обычно высокая для чистого текста)
    """
    # FastText работает лучше с одной строкой без переносов
    clean_input = text.replace('\n', ' ')

    # model.predict возвращает кортеж: (['__label__en'], [0.98])
    predictions = lang_model.predict(clean_input, k=1)
    label = predictions[0][0]
    prob = predictions[1][0]

    if label == '__label__en' and prob >= threshold:
        return True
    return False

def get_sentence_trigrams(text):
    sentences = sent_tokenize(text)
    trigrams = []
    for i in range(len(sentences) - 2):
        trigram_text = " ".join(sentences[i:i+3])
        trigrams.append(hashlib.md5(trigram_text.encode('utf-8')).hexdigest())
    return trigrams

def is_valid_line(line):
    line = line.strip()
    if line.startswith("©"):
        return False
    if not line or len(line.split()) < 5:
        return False
    if line[-1] not in {'.', '!', '?', '"', "'"}:
        return False
    return True

def clean_pipeline(text, seen_hashes, seen_trigrams, do_dedupe=True):
    # --- НОВОЕ: Проверка языка ---
    # Делаем это первым, чтобы не тратить время на очистку не-английского текста
    if not is_english(text):
        return None, seen_hashes, seen_trigrams

    # 1. Проверка уровня документа (Bad words, {, Lorem Ipsum)
    text_lower = text.lower()
    if '{' in text: return None, seen_hashes, seen_trigrams
    if LOREM_IPSUM_REGEX.search(text): return None, seen_hashes, seen_trigrams
    for word in BAD_WORDS:
        if re.search(rf'\b{re.escape(word)}\b', text_lower):
            return None, seen_hashes, seen_trigrams

    # 2. Очистка уровня строк
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        line_lower = line.lower()
        if "javascript" in line_lower: continue
        if any(pattern in line_lower for pattern in BOILERPLATE_PATTERNS): continue
        if is_valid_line(line):
            line = CITATION_REGEX.sub('', line)
            cleaned_lines.append(line)

    text = '\n'.join(cleaned_lines)
    if not text.strip(): return None, seen_hashes, seen_trigrams

    # 3. Проверка количества предложений
    sentences = sent_tokenize(text)
    if len(sentences) < 3:
        return None, seen_hashes, seen_trigrams

    # 4. Дедупликация
    if do_dedupe:
        doc_hash = hashlib.md5(text.encode('utf-8')).hexdigest()
        if doc_hash in seen_hashes: return None, seen_hashes, seen_trigrams
        seen_hashes.add(doc_hash)

        trigrams = get_sentence_trigrams(text)
        for t_hash in trigrams:
            if t_hash in seen_trigrams:
                return None, seen_hashes, seen_trigrams
        seen_trigrams.update(trigrams)

    return text, seen_hashes, seen_trigrams

In [5]:
from datasets import load_dataset
from tqdm.auto import tqdm

TOTAL_TOKENS_GOAL = 1e5

DATASETS_TO_MIX = [
    {
        "name": "HuggingFaceFW/fineweb",
        "subset": None,   # Пример конфигурации fineweb
        "proportion": 0.3          # 60% от общего объема
    },
    {
        "name": "HuggingFaceFW/fineweb-edu",
        "subset": None,   # Пример конфигурации fineweb
        "proportion": 0.5          # 60% от общего объема
    },
    {
        "name": "HuggingFaceTB/finemath",
        "subset": "finemath-4plus", # Та самая приписка
        "proportion": 0.2           # 30% от общего объема
    },
]

OUTPUT_FILENAME = "final_mixed_corpus.txt"
SHUFFLE_BUFFER = 10_000  # Размер буфера перемешивания (как в C4)
SEED = 42                # Фиксируем сид для воспроизводимости

# ==========================================
# СКРИПТ СБОРА С ШАФЛИНГОМ
# ==========================================

global_seen_hashes = set()
global_seen_trigrams = set()
total_tokens_collected = 0

print(f"Цель: собрать {TOTAL_TOKENS_GOAL} токенов.")

with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:

    for dataset_info in DATASETS_TO_MIX:
        if total_tokens_collected >= TOTAL_TOKENS_GOAL:
            print("\nОбщий лимит токенов достигнут. Остановка.")
            break

        ds_name = dataset_info["name"]
        ds_subset = dataset_info["subset"]
        proportion = dataset_info["proportion"]

        current_ds_target = int(TOTAL_TOKENS_GOAL * proportion)
        current_ds_tokens = 0

        print(f"\n>>> Обработка: {ds_name} (Config: {ds_subset})")

        try:
            # 1. Загружаем
            ds = load_dataset(
                ds_name,
                ds_subset,
                split="train",
                streaming=True,
                trust_remote_code=True
            )

            # 2. ШАФЛИМ (Важно!)
            # buffer_size=10000 означает, что скрипт берет 10к примеров в память и мешает их
            ds = ds.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)

        except Exception as e:
            print(f"Ошибка при загрузке {ds_name}: {e}")
            continue

        pbar = tqdm(desc=f"Collecting {ds_name}", unit=" docs", leave=False)

        show_example = False
        for example in ds:
            if total_tokens_collected >= TOTAL_TOKENS_GOAL:
                break
            if current_ds_tokens >= current_ds_target:
                break

            text = example.get("text")
            if not text:
                continue

            cleaned, global_seen_hashes, global_seen_trigrams = clean_pipeline(
                text,
                global_seen_hashes,
                global_seen_trigrams,
                do_dedupe=True
            )

            if cleaned:
                if not show_example:
                  print(f"{ds_name} example: {cleaned}")
                  show_example = True

                f.write(cleaned + "\n\n")

                est_tokens = len(cleaned.split()) * 1.3

                current_ds_tokens += est_tokens
                total_tokens_collected += est_tokens

                pbar.update(1)
                pbar.set_postfix_str(f"Total: {int(total_tokens_collected)}")

        pbar.close()
        print(f"Забрали из {ds_name}: ~{int(current_ds_tokens)} токенов.")

print(f"\n========================================")
print(f"Сбор завершен! Всего токенов: {int(total_tokens_collected)}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'HuggingFaceFW/fineweb' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'HuggingFaceFW/fineweb' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Цель: собрать 100000.0 токенов.

>>> Обработка: HuggingFaceFW/fineweb (Config: None)


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

HuggingFaceFW/fineweb example: This article was originally published on Rooted in Rights here. It has been adapted for the We Need Diverse Books audience.
As meetings and events in the book world continue to take place in virtual spaces as a result of the COVID-19 pandemic, accessibility is too often an afterthought. Even event organizers who normally work to make sure their in-person events are accessible seem to forget that virtual events need to be accessible for the disability community, too. Data from the Pew Research Center shows that disabled people are actually much less likely to use the internet, which may be in part because inaccessibility remains a serious barrier.
So, let’s break down this barrier. Accessibility for virtual events should be a priority and central to the planning process from the beginning.
As a starting point, think about the scope of your virtual event and what platform you plan to host it on. Disabled people are twice as likely to be unemployed or live i

...

Забрали из HuggingFaceFW/fineweb-edu: ~57553 токенов.

>>> Обработка: HuggingFaceTB/finemath (Config: finemath-4plus)


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

HuggingFaceTB/finemath example: Australian coins helps children learn to recognise, identify and describe the coins according to colour, shape and size, and the identifying icon on the tails side. Additional information is provided about the Australian animals and icons featured on the tails side of each coin. This lesson is ready to teach on the interactive whiteboard.
Australian Coins – Let’s count \$1 gives children practice in counting collections of coins to \$1.
There are three separate sections which can be used over a series of lessons.
1. Count groups of coins of the same value that equal \$1.
2. Count collections of different coins that equal \$1.
3. Make collections of coins to equal \$1.
These lessons are interactive and ready to teach on the interactive whiteboard.
• comparing the value of coins.
It is a perfect game for maths groups to follow-up lessons with Australian Coins and Australian Coins Let’s count \$1.
Забрали из HuggingFaceTB/finemath: ~12619 токенов.

Сбор зав